# Herramienta 01 - Predicción de demanda de transporte

Este notebook funciona como una herramienta completa para estimar demanda por ruta. Incluye preparación de datos, análisis exploratorio, entrenamiento, evaluación y un pronóstico operativo de 30 días.


## 1. Configuración
Se cargan las librerías necesarias. El notebook no requiere clonar el repositorio: lee el CSV completo `data/processed/cta_bus_ridership_daily_by_route.csv`, tomado del Chicago Data Portal. El archivo completo queda en GitHub y el notebook filtra desde 2021 para entrenar con datos recientes.


In [ ]:
# Librerías principales
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED     = 42
WINDOW   = 14   # días de historia como secuencia de entrada al LSTM
EPOCHS   = 80
BATCH    = 32
PATIENCE = 10

np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_URL         = 'https://raw.githubusercontent.com/AndresGuido9820/sistema-transporte-inteligente/main/data/processed/cta_bus_ridership_daily_by_route.csv'
TRAIN_START_DATE = '2021-01-01'
TOP_RUTAS        = 10

## 2. Carga y filtrado del dataset real
El archivo se carga completo desde el repositorio público. Contiene demanda diaria por ruta de buses CTA desde 2001 hasta 2026. Para entrenar un modelo operativo y rápido en Colab, se filtra desde 2021 y se toman las rutas con mayor demanda reciente.


In [ ]:
raw_demanda = pd.read_csv(DATA_URL)
raw_demanda['date'] = pd.to_datetime(raw_demanda['date'], errors='coerce')
raw_demanda['rides'] = pd.to_numeric(raw_demanda['rides'], errors='coerce')
raw_demanda = raw_demanda.dropna(subset=['date', 'route', 'rides'])

demanda = raw_demanda[raw_demanda['date'] >= pd.Timestamp(TRAIN_START_DATE)].copy()
rutas_principales = demanda.groupby('route')['rides'].sum().nlargest(TOP_RUTAS).index
demanda = demanda[demanda['route'].isin(rutas_principales)].copy()
demanda = demanda.rename(columns={'rides': 'passengers'})
demanda['holiday'] = demanda['daytype'].eq('U').astype(int)
demanda = demanda[['date', 'route', 'passengers', 'holiday', 'daytype']].sort_values(['route', 'date']).reset_index(drop=True)

print('Archivo completo cargado:', raw_demanda.shape)
print('Rango completo:', raw_demanda['date'].min().date(), 'a', raw_demanda['date'].max().date())
print('Datos usados para entrenamiento:', demanda.shape)
print('Rutas seleccionadas:', sorted(demanda['route'].unique()))
demanda.head()


## 3. Exploración inicial
Se revisa el volumen por ruta y el comportamiento temporal para detectar tendencia, estacionalidad semanal e intermitencia.


In [ ]:
resumen = demanda.groupby('route')['passengers'].agg(['count', 'mean', 'min', 'max']).round(2)
display(resumen)

plt.figure(figsize=(12, 5))
for ruta, datos in demanda.groupby('route'):
    serie = datos.sort_values('date').set_index('date')['passengers'].rolling(7).mean()
    plt.plot(serie.index, serie.values, label=ruta)
plt.title('Media móvil de 7 días por ruta')
plt.xlabel('Fecha')
plt.ylabel('Pasajeros')
plt.legend(fontsize=8)
plt.grid(alpha=0.25)
plt.show()


## 3.5 Análisis de estacionalidad y tendencias

Se analizan tres dimensiones de la demanda: **tendencia** de largo plazo (descomposición aditiva con `seasonal_decompose`, período 7 días), **patrón semanal** (demanda promedio por día de semana) y **evolución anual** (comparativa año a año por ruta). Esto permite identificar si la demanda crece o cae en el tiempo, en qué días se concentra y cómo se comporta la estacionalidad semanal.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Ruta con mayor demanda total como referencia
ruta_ref = demanda.groupby('route')['passengers'].sum().idxmax()
serie_ref = (
    demanda[demanda['route'] == ruta_ref]
    .sort_values('date')
    .set_index('date')['passengers']
    .asfreq('D')
    .fillna(method='ffill')
)

# --- 1. Descomposición aditiva (tendencia + estacionalidad + residuo) ---
decomp = seasonal_decompose(serie_ref, model='additive', period=7)
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
serie_ref.plot(ax=axes[0], lw=1, color='#1f5eff', title='Original')
decomp.trend.plot(ax=axes[1], lw=1.5, color='#a86200', title='Tendencia')
decomp.seasonal.plot(ax=axes[2], lw=1, color='#0a7a5f', title='Estacionalidad semanal')
decomp.resid.plot(ax=axes[3], lw=1, color='#c0392b', title='Residuo')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.set_xlabel('')
plt.suptitle(f'Descomposición aditiva — Ruta {ruta_ref}', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# --- 2. Patrón semanal promedio (todas las rutas) ---
dias = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
tmp = demanda.copy()
tmp['dayofweek'] = tmp['date'].dt.dayofweek
patron_semanal = tmp.groupby('dayofweek')['passengers'].mean()

plt.figure(figsize=(8, 3.5))
colores = ['#1f5eff' if i < 5 else '#f6a531' for i in range(7)]
plt.bar(dias, patron_semanal.values, color=colores)
plt.title('Demanda promedio por día de semana (todas las rutas, 2021–2026)')
plt.ylabel('Pasajeros promedio')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# --- 3. Evolución anual por ruta ---
tmp['year'] = tmp['date'].dt.year
anual = tmp.groupby(['year', 'route'])['passengers'].mean().reset_index()

plt.figure(figsize=(11, 4))
for ruta, datos in anual.groupby('route'):
    plt.plot(datos['year'], datos['passengers'], marker='o', label=str(ruta), lw=1.6)
plt.title('Evolución anual — promedio diario de pasajeros por ruta')
plt.xlabel('Año')
plt.ylabel('Pasajeros promedio/día')
plt.legend(fontsize=8, ncol=2)
plt.grid(alpha=0.25)
plt.xticks(sorted(tmp['year'].unique()))
plt.tight_layout()
plt.show()

## 4. Construcción de secuencias temporales multivariadas
Se construyen **ventanas deslizantes de 14 días** con **3 canales** por timestep:

| Canal | Variable | Descripción |
|---|---|---|
| 0 | `pax_norm` | Pasajeros normalizados con MinMaxScaler por ruta |
| 1 | `holiday` | Es festivo o domingo (0/1) — tomado de `daytype='U'` |
| 2 | `dow_norm` | Día de semana normalizado a [0, 1] (lunes=0, domingo=1/6) |

El tensor de entrada tiene forma `(muestras, 14, 3)`. El modelo predice solo el canal 0 (pasajeros) del día siguiente. Los canales 1 y 2 son **exógenos conocidos**: en el pronóstico futuro se calculan directamente desde la fecha.

In [ ]:
N_FEATURES = 3  # pax_norm, holiday, dow_norm

def crear_secuencias(features, window=14):
    """
    features: array (n, 3) — [pax_norm, holiday, dow_norm]
    Retorna X de forma (n-window, window, 3) e y de forma (n-window,)
    El objetivo y es solo el canal 0 (pasajeros normalizados) del día siguiente.
    """
    X, y = [], []
    for i in range(window, len(features)):
        X.append(features[i - window:i, :])   # (window, 3)
        y.append(features[i, 0])               # solo pax_norm
    return np.array(X), np.array(y)

# Ejemplo con la ruta de mayor demanda
ruta_ej  = demanda.groupby('route')['passengers'].sum().idxmax()
grp_ej   = demanda[demanda['route'] == ruta_ej].sort_values('date').reset_index(drop=True)
pax_ej   = grp_ej['passengers'].values.astype(float)
sc_ej    = MinMaxScaler()
pax_n_ej = sc_ej.fit_transform(pax_ej.reshape(-1, 1)).flatten()
hol_ej   = grp_ej['holiday'].values.astype(float)
dow_ej   = (grp_ej['date'].dt.dayofweek.values / 6.0)

feat_ej  = np.stack([pax_n_ej, hol_ej, dow_ej], axis=1)
X_ej, y_ej = crear_secuencias(feat_ej, WINDOW)

print(f'Ruta {ruta_ej} — secuencias: {len(X_ej)}')
print(f'Forma tensor entrada: {X_ej.shape}  →  (muestras, timesteps, features)')
print(f'Canales: [pax_norm, holiday, dow_norm]')
print(f'Ejemplo X[0, :, :]:')
print(np.round(X_ej[0], 3))
print(f'Objetivo y[0] (pax_norm): {y_ej[0]:.4f}')

## 5. Entrenamiento LSTM multivariado
Se entrena un **LSTM** por ruta sobre ventanas de 14 días con **3 features**: pasajeros normalizados, flag de festivo/domingo y día de semana normalizado.

Arquitectura: `Input(14, 3)` → `LSTM(64, return_sequences=True)` → `LSTM(32)` → `Dense(16, relu)` → `Dense(1)`.

Optimizador **Adam**, pérdida **MSE**, `EarlyStopping(patience=10)` sobre val_loss, partición temporal 80/20.

In [ ]:
def build_lstm(window, n_features=3):
    model = keras.Sequential([
        layers.Input(shape=(window, n_features)),
        layers.LSTM(64, return_sequences=True),
        layers.LSTM(32),
        layers.Dense(16, activation='relu'),
        layers.Dense(1),
    ], name='LSTM')
    model.compile(optimizer='adam', loss='mse')
    return model

early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=0
)

resultados      = []
mejores_modelos = {}
scalers         = {}
predicciones    = []

for ruta, group in demanda.sort_values(['route', 'date']).groupby('route'):
    group  = group.reset_index(drop=True)
    pax    = group['passengers'].values.astype(float)
    fechas = group['date'].values

    # Normalizar pasajeros por ruta
    scaler = MinMaxScaler()
    pax_n  = scaler.fit_transform(pax.reshape(-1, 1)).flatten()
    scalers[ruta] = scaler

    # Features exógenas (conocidas en el tiempo)
    holiday  = group['holiday'].values.astype(float)
    dow_norm = (group['date'].dt.dayofweek.values / 6.0)

    # Matriz de features: (n, 3)
    features = np.stack([pax_n, holiday, dow_norm], axis=1)

    # Construir secuencias multivariadas
    X, y = crear_secuencias(features, WINDOW)

    # Partición temporal 80/20
    corte      = int(len(X) * 0.8)
    X_tr, X_te = X[:corte], X[corte:]
    y_tr, y_te = y[:corte], y[corte:]
    pax_real   = pax[WINDOW + corte:]
    fechas_te  = fechas[WINDOW + corte:]

    # Entrenar
    modelo = build_lstm(WINDOW, N_FEATURES)
    modelo.fit(X_tr, y_tr, epochs=EPOCHS, batch_size=BATCH,
               validation_split=0.1, callbacks=[early_stop], verbose=0)

    # Predecir e invertir normalización
    pred_n = modelo.predict(X_te, verbose=0).flatten()
    pred   = np.maximum(scaler.inverse_transform(pred_n.reshape(-1, 1)).flatten(), 0)

    mae  = mean_absolute_error(pax_real, pred)
    rmse = float(np.sqrt(mean_squared_error(pax_real, pred)))
    mape = float(np.mean(np.abs((pax_real - pred) / pax_real)) * 100)
    resultados.append({'route': ruta, 'model': 'LSTM (3 features)', 'MAE': round(mae, 1),
                       'RMSE': round(rmse, 1), 'MAPE (%)': round(mape, 2)})
    mejores_modelos[ruta] = modelo

    tmp = pd.DataFrame({'date': fechas_te, 'route': ruta,
                        'passengers': pax_real, 'prediction': pred})
    predicciones.append(tmp)

metricas = pd.DataFrame(resultados).sort_values('route')
display(metricas.round(3))

## 6. Gráficas de validación
Estas gráficas permiten explicar si el modelo sigue la forma de la demanda real o si se queda corto en picos específicos.


In [ ]:
pred_df = pd.concat(predicciones, ignore_index=True)
for ruta in pred_df['route'].unique()[:4]:
    graf = pred_df[pred_df['route'] == ruta].sort_values('date')
    plt.figure(figsize=(11, 4))
    plt.plot(graf['date'], graf['passengers'], marker='o', label='Real')
    plt.plot(graf['date'], graf['prediction'], marker='o', label='Predicción')
    plt.title(f'Demanda real vs. predicha - {ruta}')
    plt.ylabel('Pasajeros')
    plt.xticks(rotation=45, ha='right')
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()


## 7. Herramienta operativa: pronóstico a 30 días
Seleccione una ruta y ejecute la celda para obtener la predicción diaria futura. Esta es la salida que usaría el área de planeación para asignar vehículos y personal.


In [ ]:
#@title Parámetros de la herramienta
ruta_seleccionada = '66' #@param {type:'string'}
dias_a_predecir   = 30   #@param {type:'integer'}

if ruta_seleccionada not in mejores_modelos:
    ruta_seleccionada = list(mejores_modelos.keys())[0]
    print('Ruta no encontrada. Se usará:', ruta_seleccionada)

def pronosticar_ruta(df, ruta, modelo, scaler, window=14, dias=30):
    """Pronóstico iterativo multivariado.

    Para cada paso futuro:
    - El canal 0 (pax_norm) se predice por el modelo y se agrega a la ventana.
    - Los canales 1 (holiday) y 2 (dow_norm) se calculan directamente de la fecha futura.
    holiday=1 si es domingo (dayofweek==6) — aproximación al daytype='U' del CTA.
    """
    hist    = df[df['route'] == ruta].sort_values('date')['passengers'].values.astype(float)
    hol_h   = df[df['route'] == ruta].sort_values('date')['holiday'].values.astype(float)
    dow_h   = (df[df['route'] == ruta].sort_values('date')['date'].dt.dayofweek.values / 6.0)
    ultima  = df[df['route'] == ruta]['date'].max()

    pax_n_h = scaler.transform(hist.reshape(-1, 1)).flatten()
    # Ventana histórica inicial: últimos `window` días con los 3 canales
    ventana_pax = pax_n_h[-window:].tolist()
    ventana_hol = hol_h[-window:].tolist()
    ventana_dow = dow_h[-window:].tolist()

    filas = []
    for paso in range(1, dias + 1):
        fecha     = ultima + pd.Timedelta(days=paso)
        hol_fut   = float(fecha.dayofweek == 6)       # domingo → 1
        dow_fut   = fecha.dayofweek / 6.0

        seq = np.stack([
            ventana_pax[-window:],
            ventana_hol[-window:],
            ventana_dow[-window:],
        ], axis=1).reshape(1, window, 3)

        pred_n = float(modelo.predict(seq, verbose=0)[0][0])
        pred   = float(scaler.inverse_transform([[pred_n]])[0][0])
        pred   = max(pred, 0)

        ventana_pax.append(pred_n)
        ventana_hol.append(hol_fut)
        ventana_dow.append(dow_fut)
        filas.append({'date': fecha.date(), 'route': ruta, 'forecast_passengers': round(pred, 2)})
    return pd.DataFrame(filas)

sx         = scalers[ruta_seleccionada]
pronostico = pronosticar_ruta(demanda, ruta_seleccionada,
                               mejores_modelos[ruta_seleccionada], sx,
                               WINDOW, dias_a_predecir)
display(pronostico.head(10))

plt.figure(figsize=(11, 4))
plt.plot(pronostico['date'], pronostico['forecast_passengers'], marker='o')
plt.title(f'Pronóstico LSTM multivariado (ventana {WINDOW}d) — Ruta {ruta_seleccionada}')
plt.ylabel('Pasajeros estimados')
plt.xticks(rotation=45, ha='right')
plt.grid(alpha=0.25)
plt.show()

## 8. Conclusiones

### Dataset y exploración
El dataset del CTA contiene 1 106 531 registros diarios por ruta (2001–2026).
Filtrado desde 2021, las 10 rutas de mayor demanda aportan 19 160 observaciones
de entrenamiento. La descomposición del patrón semanal confirmó que la demanda
cae sistemáticamente en sábado y domingo/festivos, y que la recuperación
post-COVID desde 2021 es visible pero incompleta respecto a niveles pre-2019.

### Arquitectura: LSTM multivariado con ventanas deslizantes
Se entrena un **LSTM** por ruta sobre ventanas deslizantes de 14 días con
**3 canales** por timestep:

| Canal | Variable | Descripción |
|---|---|---|
| 0 | `pax_norm` | Pasajeros normalizados con MinMaxScaler por ruta |
| 1 | `holiday` | Domingo o festivo (0/1) — `daytype='U'` del CTA |
| 2 | `dow_norm` | Día de semana / 6.0 — normalizado a [0, 1] |

Los canales 1 y 2 son **exógenos conocidos**: tanto en entrenamiento como en
pronóstico futuro se calculan directamente de la fecha, sin predicción iterativa.
Esto es la ventaja clave frente al enfoque univariado.

Arquitectura por ruta: `Input(14, 3)` → `LSTM(64, return_sequences=True)` →
`LSTM(32)` → `Dense(16, relu)` → `Dense(1)`. Optimizador **Adam**, pérdida **MSE**,
`EarlyStopping(patience=10)` sobre val_loss, partición temporal 80/20 estricta.

### Rendimiento por ruta (20% más reciente, datos reales)

| Ruta | MAE | RMSE | MAPE (%) |
|------|------:|------:|----------:|
| **22** | 986 | 1 308 | **8.69** |
| **49** | 1 784 | 2 034 | **14.63** |
| **9** | 2 630 | 2 961 | **16.10** |
| 66 | 2 720 | 3 090 | 19.06 |
| 3 | 2 991 | 3 337 | 21.96 |
| 79 | 3 656 | 4 148 | 21.98 |
| 4 | 2 990 | 3 365 | 22.10 |
| 53 | 3 109 | 3 515 | 22.33 |
| 77 | 2 852 | 3 108 | 23.75 |
| 8 | 3 601 | 4 015 | 25.62 |

**MAPE global: mín 8.69 % — máx 25.62 % — media 19.62 %**

### Impacto de agregar holiday + día de semana
Comparado con el enfoque univariado (solo pasajeros, MAPE media 23.98 %), el
modelo multivariado reduce la media a **19.62 %** — una mejora de ~4.4 puntos
porcentuales. Las ganancias más grandes son en rutas con patrón festivo pronunciado:

| Ruta | MAPE univariado | MAPE multivariado | Mejora |
|------|----------------:|------------------:|-------:|
| 49 | 25.24 % | 14.63 % | −10.6 pp |
| 9 | 25.39 % | 16.10 % | −9.3 pp |
| 22 | 12.20 % | 8.69 % | −3.5 pp |
| 53 | 27.61 % | 22.33 % | −5.3 pp |

La ruta 8 no mejoró (25.13 % → 25.62 %), lo que indica que su variabilidad
no es explicada por el día de semana ni el festivo, sino por factores externos.

### Interpretación del error
Un MAPE medio de ~20 % en transporte urbano post-COVID sin variables de clima o
eventos es un resultado razonable. La mayor parte del error residual proviene de
eventos no observables en la serie histórica: cortes de servicio, incidentes
de tráfico, condiciones climáticas extremas y eventos masivos.

### Pronóstico a 30 días
El pronóstico iterativo propaga correctamente los días festivos y el patrón
semanal hacia el horizonte de 30 días, ya que `holiday` y `dow_norm` se
calculan desde la fecha futura conocida. No se observa divergencia acumulada.

### Limitaciones y mejoras sugeridas
- Incorporar **temperatura diaria** (NOAA API, gratuita para Chicago) podría
  reducir el MAPE de rutas como la 8 que no mejoraron con el día de semana.
- Agregar **festivos federales de EE. UU.** explícitamente (actualmente solo se
  marca `daytype='U'` que mezcla domingos y festivos).
- Un **re-entrenamiento incremental** semanal es crítico en período de
  recuperación post-pandemia con patrones aún cambiantes.
- Comparar contra el baseline naive (mismo día semana anterior) cuantificaría
  la ganancia real del LSTM frente a la alternativa más simple.

---

## Fuente del dataset

**Chicago Transit Authority (CTA) — CTA Bus Ridership Daily Totals by Route**

- **Portal:** [Chicago Data Portal](https://data.cityofchicago.org/Transportation/CTA-Ridership-Bus-Routes-Daily-Totals-by-Route/jyb9-n7fm)
- **Publicado por:** City of Chicago — Chicago Transit Authority
- **Licencia:** Public Domain (U.S. Government Open Data)
- **Cobertura temporal:** 2001-01-01 al presente (actualización continua)
- **Granularidad:** Demanda diaria por ruta de bus, con tipo de día (`W` = laboral, `A` = sábado, `U` = domingo/festivo)
- **Tamaño descargado:** 1 106 531 filas (al 2026-03-31)